# Notebook 02 — SARIMA Grid Search (Part 3)

Loops over all (p,d,q) combinations with p,q ∈ [0,6] and d ∈ [0,2], selects best AIC, fits final model, and forecasts 2 years with confidence intervals.

> Runtime: ~45–90 minutes. Results are checkpointed every 10 models so you can resume if interrupted.

In [ ]:
!pip install -q pandas numpy matplotlib seaborn statsmodels scikit-learn tqdm

from google.colab import drive
drive.mount('/content/drive')

import sys
import json
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf

PROJECT_ROOT = "/content/drive/MyDrive/Assignment"
sys.path.insert(0, f"{PROJECT_ROOT}/colab")
from utils import *

paths = get_paths(PROJECT_ROOT)

weekly = pd.read_csv(paths["processed"] / "weekly_load_gw.csv",
                     parse_dates=["date"], index_col="date")["load_gw"]
train, test = train_test_split_series(weekly, TEST_WEEKS)
print(f"Train: {len(train)} weeks | Test: {len(test)} weeks")

Mounted at /content/drive
Train: 197 weeks | Test: 104 weeks


In [ ]:
# SARIMA grid search — required (p,d,q) loop
# Seasonal order fixed at (1,1,1,52) based on EDA
SEASONAL_ORDER = (1, 1, 1, SEASONALITY)
checkpoint = paths["checkpoints"] / "sarima_grid_results.csv"

# Skip grid search if already completed
best_params_path = paths["checkpoints"] / "sarima_best_params.json"

if best_params_path.exists():
    print("Loading previously saved best parameters...")
    with open(best_params_path) as f:
        best_params = json.load(f)
    grid_results = pd.read_csv(checkpoint)
    best_params["order"] = tuple(best_params["order"])
    best_params["seasonal_order"] = tuple(best_params["seasonal_order"])
else:
    grid_results, best_params = sarima_grid_search(
        train,
        seasonal_order=SEASONAL_ORDER,
        p_range=range(0, 7),
        d_range=range(0, 3),
        q_range=range(0, 7),
        checkpoint_path=checkpoint,
    )
    save_json(best_params, best_params_path)

print("\nBest model by AIC:")
print(best_params)
print(f"\nConverged models: {grid_results['converged'].sum()} / {len(grid_results)}")
grid_results.sort_values("AIC").head(10)

SARIMA grid search:   0%|          | 0/147 [00:00<?, ?it/s]


Best model by AIC:
{'order': (4, 0, 6), 'seasonal_order': (1, 1, 1, 52), 'AIC': np.float64(302.72811262819505)}

Converged models: 147 / 147


,p,d,q,AIC,seasonal_order,converged
90,4,0,6,302.728113,"(1, 1, 1, 52)",True
34,1,1,6,303.497510,"(1, 1, 1, 52)",True
97,4,1,6,303.593755,"(1, 1, 1, 52)",True
111,5,0,6,303.671401,"(1, 1, 1, 52)",True
48,2,0,6,303.939114,"(1, 1, 1, 52)",True
118,5,1,6,304.785740,"(1, 1, 1, 52)",True
76,3,1,6,304.954380,"(1, 1, 1, 52)",True
132,6,0,6,305.274087,"(1, 1, 1, 52)",True
6,0,0,6,305.674879,"(1, 1, 1, 52)",True
27,1,0,6,307.674994,"(1, 1, 1, 52)",True


In [ ]:
from pathlib import Path
import pandas as pd
ckpt = Path("/content/drive/MyDrive/Assignment/outputs/checkpoints/sarima_grid_results.csv")
print("exists:", ckpt.exists())
if ckpt.exists():
    df = pd.read_csv(ckpt)
    print("rows saved:", len(df), "/ 147")
    print(df.dropna(subset=["AIC"]).sort_values("AIC").head(3))

In [ ]:
# Fit best SARIMA model and residual diagnostics
sarima_fit = fit_sarimax(
    train,
    order=best_params["order"],
    seasonal_order=best_params["seasonal_order"],
)

residuals = sarima_fit.resid.dropna()

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(residuals.index, residuals, linewidth=0.8)
axes[0].set_title("SARIMA residuals")
axes[0].axhline(0, color="black", linewidth=0.8)

axes[1].hist(residuals, bins=30, edgecolor="white", color="steelblue")
axes[1].set_title("Residual distribution")

plot_acf(residuals, lags=40, ax=axes[2])
axes[2].set_title("Residual ACF")

plt.tight_layout()
plot_and_save(fig, paths["figures"] / "05_sarima_residuals.png")
plt.show()

In [ ]:
# Forecast with confidence intervals
sarima_fc = forecast_sarimax(sarima_fit, len(test), test.index)

metrics_sarima = evaluate_forecast("sarima", test, sarima_fc["mean"], train)

# Load benchmark metrics for comparison
bench = pd.read_csv(paths["metrics"] / "02_benchmark_metrics.csv")
all_metrics = pd.concat([bench, pd.DataFrame([metrics_sarima])], ignore_index=True)
all_metrics = all_metrics.sort_values("MASE").reset_index(drop=True)
save_metrics(all_metrics.to_dict("records"), paths["metrics"] / "03_sarima_metrics.csv")
print(all_metrics.round(3).to_string(index=False))

In [ ]:
# SARIMA forecast plot with confidence intervals
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(train.index, train, label="Training", color="steelblue")
ax.plot(test.index, test, label="Test (actual)", color="black", linewidth=2)
ax.plot(test.index, sarima_fc["mean"], label="SARIMA", color="crimson", linewidth=1.5)

if "ci_95" in sarima_fc:
    ci95 = sarima_fc["ci_95"]
    ax.fill_between(test.index, ci95.iloc[:, 0], ci95.iloc[:, 1],
                    alpha=0.15, color="crimson", label="95% CI")
if "ci_80" in sarima_fc:
    ci80 = sarima_fc["ci_80"]
    ax.fill_between(test.index, ci80.iloc[:, 0], ci80.iloc[:, 1],
                    alpha=0.25, color="crimson", label="80% CI")

ax.set_title(f"SARIMA{best_params['order']}x{best_params['seasonal_order']} forecast")
ax.set_ylabel("Average load (GW)")
ax.legend()
plt.tight_layout()
plot_and_save(fig, paths["figures"] / "06_sarima_forecast.png")
plt.show()

# Save forecasts
fc_df = pd.DataFrame({"actual": test, "sarima": sarima_fc["mean"]})
fc_df.to_csv(paths["forecasts"] / "sarima_forecast.csv")

free_memory()